In [ ]:
import json
import pandas as pd
import requests
from difflib import SequenceMatcher

# 1. Load archetype JSON dan topic keywords mapping
authority_json_path = "karakteristik_arketipe_enriched.json"
topic_kw_path = "daftar_topik_keywords_automerged.csv"

with open(authority_json_path, "r", encoding="utf-8") as f:
    archetypes = json.load(f)

topic_kw_df = pd.read_csv(topic_kw_path)
topic_mapping = {int(r['topic']): r['keywords'].split(', ')[:3] for _, r in topic_kw_df.iterrows()}

# 2. Konfigurasi API
API_KEY = "YOUR_API_KEY"  # ganti dengan API key Anda
ENDPOINT = "https://api.groq.com/openai/v1/chat/completions"

# 3. Build improved prompt dengan aturan "interpretasi unik"
system_msg = {
    "role": "system",
    "content": (
        "Anda adalah seorang Analis Data berpengalaman dan pakar perilaku pemain game. "
        "Tugas Anda adalah menafsirkan data JSON untuk setiap arketipe pemain berdasarkan fitur kuantitatif "
        "dan topik dominan yang dihasilkan dari analisis review game.\n\n"
        "FORMAT OUTPUT:\n"
        "Jawaban harus berupa array JSON tanpa teks tambahan. "
        "Setiap elemen berisi:\n"
        "  - \"arketipe\": nama key arketipe.\n"
        "  - \"fitur_kuantitatif\": ringkasan angka aktual.\n"
        "  - \"topik_dominan\": interpretasi makna topik, bukan sekadar kata kunci.\n"
        "  - \"interpretasi\": 1–2 kalimat berbahasa Indonesia yang mencakup angka aktual, makna topik, dan opini karakter pemain.\n\n"
        "ATURAN PENTING:\n"
        "- Setiap 'interpretasi' harus unik, tidak boleh sama persis atau terlalu mirip dengan arketipe lain.\n"
        "- Variasikan fokus insight (misalnya: kompetitif, sosial, eksploratif, artistik, nostalgia, dll).\n"
        "- Gunakan bahasa alami yang kaya sinonim.\n\n"
        "MAPPING TOPIK:\n"
        f"{json.dumps(topic_mapping, ensure_ascii=False)}\n\n"
        "RESPOND ONLY WITH A JSON ARRAY, tanpa teks tambahan."
    )
}

# 4. User message
user_msg = {"role": "user", "content": json.dumps(archetypes, ensure_ascii=False)}

# 5. Kirim request
payload = {
    "model": "llama3-8b-8192",
    "messages": [system_msg, user_msg],
    "temperature": 0.7,
    "max_tokens": 1024
}
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
response = requests.post(ENDPOINT, headers=headers, json=payload)
response.raise_for_status()
raw = response.json()["choices"][0]["message"]["content"].strip()

# 6. Extract JSON
start = raw.find('[')
end = raw.rfind(']')
json_text = raw[start:end+1] if start != -1 and end != -1 else raw

try:
    parsed = json.loads(json_text)
except json.JSONDecodeError:
    print("⚠️ Parse JSON gagal. Raw output:\n", raw)
    raise

# 7. Post-processing: pastikan interpretasi unik
def is_similar(a, b, threshold=0.85):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio() > threshold

unique_interpretations = []
for item in parsed:
    interp = item["interpretasi"]
    # Jika terlalu mirip dengan sebelumnya, tambahkan variasi
    if any(is_similar(interp, seen) for seen in unique_interpretations):
        item["interpretasi"] += " Karakter pemain ini juga memiliki kecenderungan unik yang membedakannya."
    unique_interpretations.append(item["interpretasi"])

# 8. Simpan dan print
with open("interpretasi_arketipe.json", "w", encoding="utf-8") as f:
    json.dump(parsed, f, ensure_ascii=False, indent=4)

print("✅ JSON interpretasi disimpan di 'interpretasi_arketipe.json'")
print(json.dumps(parsed, ensure_ascii=False, indent=4))


✅ JSON interpretasi disimpan di 'interpretasi_arketipe.json'
[
    {
        "arketipe": "Arketype 1",
        "fitur_kuantitatif": {
            "average_game_owned": 226.58,
            "average_playtime": 25.61,
            "average_achievement": 2456.17
        },
        "topik_dominan": "Pengalaman game yang unik dan menarik",
        "interpretasi": "Pemain ini memiliki rata-rata 226 game dan bermain selama 25.61 jam, serta memiliki 2456 achievement. Mereka lebih memilih game yang unik dan menarik, seperti game pinball yang menawarkan pengalaman berbeda."
    },
    {
        "arketipe": "Arketype 2",
        "fitur_kuantitatif": {
            "average_game_owned": 231.29,
            "average_playtime": 35.03,
            "average_achievement": 2607.86
        },
        "topik_dominan": "Kesulitan teknis dan masalah game",
        "interpretasi": "Pemain ini memiliki rata-rata 231 game dan bermain selama 35.03 jam, serta memiliki 2607 achievement. Mereka lebih mengalami kesuli